# Notebook 3: Sentence embeddings (SBERT zero-shot, cross-encoder, fine-tuned)

**Before running:** turn on the free GPU with **Runtime → Change runtime type → T4 GPU → Save**.

This notebook adds the **semantic** methods to the results table:

| Group | Models | Trained on our data? |
|---|---|---|
| Zero-shot bi-encoders | MiniLM-L6, MPNet-base, BGE-base, E5-base | No |
| Long-context bi-encoder | Nomic-embed v1.5 (reads 8,192 tokens) | No |
| Chunking variant | Each short model + resume split into 200-word chunks (max score) | No |
| Zero-shot cross-encoder | MS-MARCO MiniLM cross-encoder | No |
| **Fine-tuned** | BGE-base + CoSENT loss on the train split (**standard**) | Yes |
| **Fine-tuned, strict** | Same, but training excludes every resume that appears in test | Yes |

**Why the strict version:** 430 of the 434 test resumes also appear in the training pairs (with different jobs). The strict run shows how much of the fine-tuned score comes from having seen those resumes before.

**Latency** for bi-encoders is the **query-time** cost: embed the job, then compare it with resume embeddings that were computed in advance, as a real portal would. The one-off indexing time per resume is reported separately. The cross-encoder must read every job–resume pair at query time.

Expected run time on a T4: about 30–40 minutes (fine-tuning is most of it).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip -q install -U sentence-transformers einops

import re, time, json, os, gc
import numpy as np
import pandas as pd
import torch

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
device = 'cuda'
print(torch.cuda.get_device_name(0))

DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

train_df = pd.read_parquet(f'{DATA}/train.parquet')
val_df = pd.read_parquet(f'{DATA}/val.parquet')
test_df = pd.read_parquet(f'{DATA}/test.parquet')
print(len(train_df), len(val_df), len(test_df))

## 1. Metrics and evaluation loop (identical to Notebook 2)

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))

def ndcg_at_k(ranked_rels, k):
    ideal = dcg(sorted(ranked_rels, reverse=True)[:k])
    return dcg(ranked_rels[:k]) / ideal if ideal > 0 else 0.0

def precision_at_k(ranked_rels, k, min_rel=2):
    return np.mean([r >= min_rel for r in ranked_rels[:k]])

def mrr(ranked_rels, min_rel=2):
    for i, r in enumerate(ranked_rels):
        if r >= min_rel:
            return 1.0 / (i + 1)
    return 0.0

def score_query(ranked_rels):
    return {'NDCG@5': ndcg_at_k(ranked_rels, 5), 'NDCG@10': ndcg_at_k(ranked_rels, 10),
            'P@5': precision_at_k(ranked_rels, 5), 'MRR': mrr(ranked_rels)}

def bootstrap_ci(values, n=2000, seed=SEED):
    rng = np.random.default_rng(seed)
    values = np.asarray(values)
    means = [rng.choice(values, len(values), replace=True).mean() for _ in range(n)]
    return np.percentile(means, [2.5, 97.5])

def evaluate(method_name, score_fn, df, split_name='test'):
    rng = np.random.default_rng(SEED)
    rows, latencies = [], []
    for job_id, group in df[df.eval_usable].groupby('job_id'):
        job_text = group.job_text.iloc[0]
        t0 = time.perf_counter()
        scores = np.asarray(score_fn(job_text, group.resume_text.tolist()), dtype=float)
        latencies.append((time.perf_counter() - t0) * 1000)
        order = np.lexsort((rng.random(len(scores)), -scores))
        ranked_rels = group.relevance.to_numpy()[order].tolist()
        rows.append({'method': method_name, 'split': split_name, 'job_id': job_id,
                     'candidates': len(group), **score_query(ranked_rels)})
    per_query = pd.DataFrame(rows)
    summary = {'method': method_name, 'split': split_name, 'jobs': len(per_query)}
    for m in ['NDCG@5', 'NDCG@10', 'P@5', 'MRR']:
        lo, hi = bootstrap_ci(per_query[m])
        summary[m] = round(per_query[m].mean(), 4)
        summary[m + ' CI'] = f'[{lo:.3f}, {hi:.3f}]'
    summary['ms/job (median)'] = round(float(np.median(latencies)), 2)
    return summary, per_query

summaries, per_query_all = [], []
def run_on_val_and_test(name, score_fn, extra=None):
    for split_name, df in [('val', val_df), ('test', test_df)]:
        s, pq = evaluate(name, score_fn, df, split_name)
        s.update(extra or {})
        summaries.append(s); per_query_all.append(pq)
        print(f"{name:32} {split_name:4} NDCG@10 {s['NDCG@10']:.4f} {s['NDCG@10 CI']}  P@5 {s['P@5']:.3f}  MRR {s['MRR']:.3f}  {s['ms/job (median)']} ms/job")

## 2. Scorer for bi-encoders
Resumes are embedded once in advance (the "index"). At query time only the job is embedded. With **chunking**, a resume is split into 200-word chunks (150-word stride), and its score is the best-matching chunk.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

eval_resumes = pd.concat([val_df, test_df]).resume_text.unique().tolist()

def chunk_words(text, size=200, stride=150):
    words = text.split()
    if len(words) <= size:
        return [text]
    return [' '.join(words[i:i + size]) for i in range(0, len(words) - size + stride, stride)]

def make_scorer(model, query_prefix='', doc_prefix='', chunked=False, batch_size=32):
    def enc(texts, prefix):
        return model.encode([prefix + t for t in texts], batch_size=batch_size,
                            normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)
    t0 = time.perf_counter()
    pieces = {r: (chunk_words(r) if chunked else [r]) for r in eval_resumes}
    flat = [p for r in eval_resumes for p in pieces[r]]
    emb = enc(flat, doc_prefix)
    index, i = {}, 0
    for r in eval_resumes:
        n = len(pieces[r]); index[r] = emb[i:i + n]; i += n
    ms_per_resume = (time.perf_counter() - t0) * 1000 / len(eval_resumes)

    def score(job_text, candidates):
        q = enc([job_text], query_prefix)[0]
        return [float((index[c] @ q).max()) for c in candidates]
    return score, round(ms_per_resume, 2)

def free(model):
    del model; gc.collect(); torch.cuda.empty_cache()

## 3. Zero-shot bi-encoders (truncated and chunked)
Model choice is fixed in advance (not tuned on test): two classic SBERT models and two strong retrieval models. Prefixes follow each model's documentation.

In [ ]:
ZERO_SHOT = [
    # name, Hugging Face id, query prefix, document prefix, max tokens
    ('MiniLM-L6',  'sentence-transformers/all-MiniLM-L6-v2', '', '', 256),
    ('MPNet-base', 'sentence-transformers/all-mpnet-base-v2', '', '', 384),
    ('BGE-base',   'BAAI/bge-base-en-v1.5', '', '', 512),
    ('E5-base',    'intfloat/e5-base-v2', 'query: ', 'passage: ', 512),
]
for name, hf_id, qp, dp, max_len in ZERO_SHOT:
    model = SentenceTransformer(hf_id, device=device)
    model.max_seq_length = max_len
    for chunked in (False, True):
        label = f'{name} (zero-shot{", chunked" if chunked else ""})'
        scorer, index_ms = make_scorer(model, qp, dp, chunked)
        run_on_val_and_test(label, scorer, {'index ms/resume': index_ms})
    free(model)

## 4. Long-context bi-encoder (Nomic-embed v1.5, 8,192 tokens)
Reads the whole resume without truncation. If this model fails to load in your Colab version, the cell prints a note and the rest of the notebook still runs.

In [ ]:
try:
    model = SentenceTransformer('nomic-ai/nomic-embed-text-v1.5', device=device, trust_remote_code=True)
    model.max_seq_length = 2048  # covers the longest resumes (about 3,100 words) while fitting T4 memory
    scorer, index_ms = make_scorer(model, 'search_query: ', 'search_document: ', False, batch_size=8)
    run_on_val_and_test('Nomic-v1.5 (zero-shot, long)', scorer, {'index ms/resume': index_ms})
    free(model)
except Exception as e:
    print('Nomic model skipped:', type(e).__name__, e)

## 5. Zero-shot cross-encoder
Reads the job and the resume **together**, which is usually more accurate but must run on every pair at query time, so it's slower. Input is limited to 512 tokens shared between job and resume.

In [ ]:
ce = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=device, max_length=512)
def ce_scores(job_text, candidates):
    return ce.predict([(job_text, c) for c in candidates], batch_size=32, show_progress_bar=False)
run_on_val_and_test('CrossEnc-MiniLM (zero-shot)', ce_scores)
free(ce)

## 6. Fine-tuned BGE-base (CoSENT loss)
- Base model **BGE-base**, chosen in advance because it is a strong open retrieval model that reads 512 tokens.
- Training pairs: job + resume, target score = relevance / 2 (so 0, 0.5, 1).
- **CoSENT loss** learns to *order* pairs by score, which matches our ranking goal.
- 2 epochs, learning rate 2e-5, batch 16, 10% warm-up, fp16, seed 42. **Fixed in advance, not tuned on test.**
- Validation scores are printed so you can see the model isn't over-fitting.

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments, losses

def finetune(pairs, tag, base='BAAI/bge-base-en-v1.5'):
    model = SentenceTransformer(base, device=device)
    model.max_seq_length = 512
    ds = Dataset.from_dict({
        'sentence1': pairs.job_text.tolist(),
        'sentence2': pairs.resume_text.tolist(),
        'score': (pairs.relevance / 2).astype(float).tolist(),
    }).shuffle(seed=SEED)
    args = SentenceTransformerTrainingArguments(
        output_dir=f'/content/ft-{tag}', num_train_epochs=2, per_device_train_batch_size=16,
        learning_rate=2e-5, warmup_ratio=0.1, fp16=True, seed=SEED,
        save_strategy='no', logging_steps=50, report_to='none')
    trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=ds,
                                         loss=losses.CoSENTLoss(model))
    t0 = time.perf_counter()
    trainer.train()
    print(f'{tag}: trained on {len(pairs)} pairs in {(time.perf_counter() - t0) / 60:.1f} min')
    return model

### 6a. Standard: all training pairs

In [ ]:
model = finetune(train_df, 'standard')
scorer, index_ms = make_scorer(model)
run_on_val_and_test('BGE-base (fine-tuned)', scorer, {'index ms/resume': index_ms, 'train pairs': len(train_df)})
model.save(f'{RESULTS}/../models/bge-base-finetuned')  # kept on Drive for the portal demo
free(model)

### 6b. Strict: no test resume is seen in training
Only the **test** result is meaningful here (the validation resumes are not excluded).

In [ ]:
test_resumes = set(test_df.resume_id)
strict_train = train_df[~train_df.resume_id.isin(test_resumes)]
print(f'strict training set: {len(strict_train)} pairs, {strict_train.resume_id.nunique()} resumes, '
      f'{strict_train.job_id.nunique()} jobs, labels {strict_train.label.value_counts().to_dict()}')

model = finetune(strict_train, 'strict')
scorer, index_ms = make_scorer(model)
s, pq = evaluate('BGE-base (fine-tuned, strict)', scorer, test_df, 'test')
s.update({'index ms/resume': index_ms, 'train pairs': len(strict_train)})
summaries.append(s); per_query_all.append(pq)
print(s)
free(model)

## 7. Results table (with the Notebook 2 baselines)

In [ ]:
base = pd.read_csv(f'{RESULTS}/02_baselines_summary.csv')
emb = pd.DataFrame(summaries)
table = pd.concat([base, emb], ignore_index=True)
cols = ['method', 'split', 'jobs', 'NDCG@5', 'NDCG@10', 'NDCG@10 CI', 'P@5', 'MRR', 'ms/job (median)', 'index ms/resume']
test_table = table[table.split == 'test'][cols].sort_values('NDCG@10', ascending=False)
test_table

## 8. Significance against TF-IDF (paired permutation test, Holm-corrected)
Each method is compared with the strongest baseline (TF-IDF) on NDCG@10 per job. Holm correction keeps the error rate at 5% across all comparisons.

In [ ]:
def paired_permutation_p(a, b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    diff = np.asarray(a) - np.asarray(b)
    signs = rng.choice([-1, 1], size=(n, len(diff)))
    return float((np.abs((signs * diff).mean(axis=1)) >= abs(diff.mean())).mean())

base_pq = pd.read_csv(f'{RESULTS}/02_baselines_per_query.csv')
all_pq = pd.concat([base_pq, pd.concat(per_query_all)], ignore_index=True)
wide = all_pq[all_pq.split == 'test'].pivot(index='job_id', columns='method', values='NDCG@10')

tests = []
for m in wide.columns:
    if m in ('TF-IDF', 'Random'):
        continue
    both = wide[[m, 'TF-IDF']].dropna()
    tests.append({'method': m, 'jobs': len(both), 'diff vs TF-IDF': round((both[m] - both['TF-IDF']).mean(), 4),
                  'p': paired_permutation_p(both[m], both['TF-IDF'])})
sig = pd.DataFrame(tests).sort_values('p').reset_index(drop=True)
k = len(sig)
sig['p (Holm)'] = np.minimum(1, np.maximum.accumulate([(k - i) * p for i, p in enumerate(sig.p)]))
sig['significant'] = sig['p (Holm)'] < 0.05
sig.sort_values('diff vs TF-IDF', ascending=False)

## 9. Save

In [ ]:
emb.to_csv(f'{RESULTS}/03_embeddings_summary.csv', index=False)
pd.concat(per_query_all).to_csv(f'{RESULTS}/03_embeddings_per_query.csv', index=False)
table.to_csv(f'{RESULTS}/results_table_so_far.csv', index=False)
sig.to_csv(f'{RESULTS}/03_significance_vs_tfidf.csv', index=False)
print(test_table[['method', 'NDCG@10', 'P@5', 'MRR', 'ms/job (median)']].to_string(index=False))

## Done
Saved to `MyDrive/resume-job-matching/results/`: the summary, per-job scores, the combined results table and the significance tests. The fine-tuned model is saved to `MyDrive/resume-job-matching/models/` for the portal demo.

**Next:** Notebook 4, LLM re-ranking of the top candidates, with cost and latency.